In [ ]:
import os
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from PIL import Image

# 1. Auto-discover the Kaggle frames path
frames_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'frames' in dirs:
        frames_path = os.path.join(root, 'frames')
        break

if not frames_path:
    raise FileNotFoundError("Could not find 'frames'. Make sure FakeAVCeleb is attached.")

print(f"✅ Found frames at: {frames_path}")

# 2. Custom Dataset for Video Sequences
class VideoSequenceDataset(Dataset):
    def __init__(self, root_dir, sequence_length=10, transform=None):
        self.root_dir = root_dir
        self.sequence_length = sequence_length
        self.transform = transform
        self.video_folders = []
        self.labels = []

        print("Scanning directory for valid video sequences...")
        
        for root_path, dirs, files in os.walk(root_dir):
            jpg_files = sorted([f for f in files if f.endswith(('.jpg', '.png'))])
            if len(jpg_files) >= self.sequence_length:
                self.video_folders.append(root_path)
                
                # Labeling: 1 for Fake (contains FakeVideo), 0 for Real
                if 'FakeVideo' in root_path:
                    self.labels.append(1.0)
                else:
                    self.labels.append(0.0)

        print(f"✅ Found {len(self.video_folders)} valid video sequences.")

    def __len__(self):
        return len(self.video_folders)

    def __getitem__(self, idx):
        video_path = self.video_folders[idx]
        label = self.labels[idx]
        
        frames = sorted([f for f in os.listdir(video_path) if f.endswith(('.jpg', '.png'))])
        
        # Grab evenly spaced frames to cover the whole video
        step = max(1, len(frames) // self.sequence_length)
        selected_frames = frames[0:self.sequence_length*step:step][:self.sequence_length]
        
        sequence_tensor = []
        for frame_name in selected_frames:
            img_path = os.path.join(video_path, frame_name)
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            sequence_tensor.append(image)
            
        return torch.stack(sequence_tensor), torch.tensor([label], dtype=torch.float32)

# 3. Transforms and DataLoaders
seq_transforms = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

full_dataset = VideoSequenceDataset(root_dir=frames_path, sequence_length=10, transform=seq_transforms)

# Subset to simulate 400 videos to keep training times reasonable
subset_size = min(400, len(full_dataset)) 
indices = list(range(subset_size))
np.random.seed(42)
np.random.shuffle(indices)

split = int(np.floor(0.2 * subset_size))
train_idx, val_idx = indices[split:], indices[:split]

# ⚠️ CRITICAL: Batch size MUST be small (4) because 1 batch = 4 videos * 10 frames = 40 total images
train_loader = DataLoader(Subset(full_dataset, train_idx), batch_size=4, shuffle=True)
val_loader = DataLoader(Subset(full_dataset, val_idx), batch_size=4, shuffle=False)

print(f"✅ Dataloaders ready: {len(train_idx)} Train sequences, {len(val_idx)} Val sequences.")

cell 1

In [ ]:
import os
import torch
import numpy as np
import torch.nn as nn
import timm
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import copy
import time

# --- 1. SETUP & HARDWARE ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Initializing Final Production Run on: {device}")

# Auto-discover paths
frames_path, weights_path = None, None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'frames' in dirs and not frames_path:
        frames_path = os.path.join(root, 'frames')
    if 'best_deepfake_model.pth' in files and not weights_path:
        weights_path = os.path.join(root, 'best_deepfake_model.pth')

if not frames_path or not weights_path:
    raise FileNotFoundError("Missing dataset or Phase 1 weights! Check your Kaggle inputs.")

# --- 2. PRODUCTION DATALOADER ---
class FinalVideoDataset(Dataset):
    def __init__(self, root_dir, sequence_length=10, transform=None):
        self.sequence_length = sequence_length
        self.transform = transform
        self.video_folders = []
        self.labels = []

        print("Scanning full dataset for sequences...")
        for root_path, dirs, files in os.walk(root_dir):
            jpg_files = sorted([f for f in files if f.endswith(('.jpg', '.png'))])
            if len(jpg_files) >= self.sequence_length:
                self.video_folders.append(root_path)
                self.labels.append(1.0 if 'FakeVideo' in root_path else 0.0)

        print(f"✅ Locked in {len(self.video_folders)} total video sequences for final training.")

    def __len__(self): return len(self.video_folders)

    def __getitem__(self, idx):
        video_path = self.video_folders[idx]
        label = self.labels[idx]
        frames = sorted([f for f in os.listdir(video_path) if f.endswith(('.jpg', '.png'))])
        
        step = max(1, len(frames) // self.sequence_length)
        selected_frames = frames[0:self.sequence_length*step:step][:self.sequence_length]
        
        sequence_tensor = [self.transform(Image.open(os.path.join(video_path, f)).convert('RGB')) for f in selected_frames]
        return torch.stack(sequence_tensor), torch.tensor([label], dtype=torch.float32)

seq_transforms = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

full_dataset = FinalVideoDataset(root_dir=frames_path, sequence_length=10, transform=seq_transforms)

# 80/20 Split on the FULL dataset
dataset_size = len(full_dataset)
indices = list(range(dataset_size))
np.random.seed(42)
np.random.shuffle(indices)
split = int(np.floor(0.2 * dataset_size))
train_idx, val_idx = indices[split:], indices[:split]

# Optimized Dataloaders for maximum speed
train_loader = DataLoader(Subset(full_dataset, train_idx), batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(Subset(full_dataset, val_idx), batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

# --- 3. THE FINAL ARCHITECTURE ---
class TemporalDeepfakeDetector(nn.Module):
    def __init__(self):
        super(TemporalDeepfakeDetector, self).__init__()
        self.backbone = timm.create_model('xception', pretrained=False, num_classes=0)
        self.lstm = nn.LSTM(input_size=self.backbone.num_features, hidden_size=512, num_layers=1, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 1), nn.Sigmoid()
        )

    def forward(self, x):
        batch_size, seq_length, c, h, w = x.size()
        x = x.view(batch_size * seq_length, c, h, w)
        features = self.backbone.forward_features(x)
        features = nn.functional.adaptive_avg_pool2d(features, (1, 1)).view(features.size(0), -1) 
        lstm_out, _ = self.lstm(features.view(batch_size, seq_length, -1))
        return self.classifier(lstm_out[:, -1, :])

final_model = TemporalDeepfakeDetector().to(device)
final_model.load_state_dict(torch.load(weights_path, map_location=device), strict=False)

# Freeze spatial, train temporal
for param in final_model.backbone.parameters(): param.requires_grad = False
print("✅ Final Architecture built. Phase 1 weights loaded. Ready for training.")

cell 2

In [ ]:
import time
import copy
from tqdm import tqdm
import torch.optim as optim
import torch.nn as nn
import torch

criterion = nn.BCELoss()

# Only train the LSTM and classifier, backbone is frozen
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, final_model.parameters()), lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2, factor=0.5)

num_epochs = 10
best_model_wts = copy.deepcopy(final_model.state_dict())
best_val_loss = float('inf')

train_losses, val_losses = [], []
train_accs, val_accs = [], []

print("🚀 Starting Final Phase 2 Production Training...")
start_time = time.time()

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print('-' * 10)
    
    # --- TRAINING ---
    final_model.train()
    running_loss, running_corrects = 0.0, 0
    
    for inputs, labels in tqdm(train_loader, desc="Training LSTM"):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = final_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        preds = (outputs > 0.5).float()
        running_corrects += torch.sum(preds == labels.data)
        
    epoch_train_loss = running_loss / len(train_loader.dataset)
    epoch_train_acc = running_corrects.double() / len(train_loader.dataset)
    train_losses.append(epoch_train_loss)
    train_accs.append(epoch_train_acc.item())
    
    # --- VALIDATION ---
    final_model.eval()
    val_loss, val_corrects = 0.0, 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Validating LSTM"):
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = final_model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            preds = (outputs > 0.5).float()
            val_corrects += torch.sum(preds == labels.data)
            
    epoch_val_loss = val_loss / len(val_loader.dataset)
    epoch_val_acc = val_corrects.double() / len(val_loader.dataset)
    val_losses.append(epoch_val_loss)
    val_accs.append(epoch_val_acc.item())
    
    scheduler.step(epoch_val_loss)
    
    print(f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f}")
    print(f"Val Loss:   {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")
    
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_model_wts = copy.deepcopy(final_model.state_dict())
        torch.save(final_model.state_dict(), 'production_temporal_lstm_model.pth')
        print("⭐ Saved Production LSTM weights.")
        
    # Crucial for full dataset runs: clear out dead tensors
    torch.cuda.empty_cache()

time_elapsed = time.time() - start_time
print(f'\n✅ Production Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
final_model.load_state_dict(best_model_wts)

cell 3